**Load & Explore Dataset**

In [1]:
import pandas as pd

# Load the dataset (tab-separated file)
df = pd.read_csv("spam_dataset.tsv", sep='\t', header=None, names=["label", "message"])

# Show first 5 rows
print("Sample data:")
print(df.head())

# Check dataset size and balance
print("\nDataset shape:", df.shape)
print("\nLabel distribution:")
print(df['label'].value_counts())


Sample data:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


**Preprocessing & Splitting data**

In [2]:
from sklearn.model_selection import train_test_split

# Encode labels: ham → 0, spam → 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Features and target
X = df['message']
y = df['label_num']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Train set size: 4457
Test set size: 1115


**Text Vectorization (TF-IDF)**

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit on train data and transform both train & test
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (4457, 5000)
Test TF-IDF shape: (1115, 5000)


**Training SVM Model**

In [5]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize SVM model
svm_model = LinearSVC()

# Train the model
svm_model.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = svm_model.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Test Accuracy: 0.9811659192825112

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.87      0.92       149

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
 [[965   1]
 [ 20 129]]


**Make It Interactive (Predict Any SMS)**

In [ ]:
def predict_sms(text):
    # Transform input text using the same TF-IDF vectorizer
    text_tfidf = vectorizer.transform([text])
    # Predict using trained SVM model
    prediction = svm_model.predict(text_tfidf)[0]
    return "Spam" if prediction == 1 else "Ham"

# Example usage
while True:
    msg = input("\nEnter an SMS to check (or type 'exit' to quit): ")
    if msg.lower() == 'exit':
        break
    print("Prediction:", predict_sms(msg))



Enter an SMS to check (or type 'exit' to quit):  "Congratulations! You won a free iPhone!


Prediction: Spam



Enter an SMS to check (or type 'exit' to quit):  Hey, are we meeting tomorrow?


Prediction: Ham



Enter an SMS to check (or type 'exit' to quit):  Congratulations! You won a iPhone!


Prediction: Ham



Enter an SMS to check (or type 'exit' to quit):  Congratulations! You won an iPhone!"


Prediction: Ham
